# Soccer Forecast Agent — Final Submission

**Multi-Agent AI System for Premier League Match Forecasting and Alert Generation**

---

This notebook demonstrates the complete end-to-end *Soccer Forecast Agent*: a production-grade
multi-agent AI system that monitors Premier League fixtures, computes probabilistic forecasts
using a Dixon-Coles statistical model enhanced with recency-weighted form, gathers qualitative
news evidence via a ReAct research loop, and fires styled HTML email alerts when a statistically
meaningful edge is detected over market-implied probabilities.

**Stack:** Python 3.11 · LangGraph · OpenAI API · SQLite · ChromaDB · SentenceTransformers · The Odds API · Tavily Search

> **Note — API keys required to execute the live pipeline cells.**
> Cells that call external APIs are marked with `# [LIVE]`.
> All other cells run offline against the local SQLite / ChromaDB stores.


---

## System Architecture

The system follows a supervisor-based multi-agent pattern with four specialised agents orchestrated by a
[LangGraph](https://github.com/langchain-ai/langgraph) `StateGraph`. All agents depend only on
abstract `Protocol` interfaces — no concrete SDK import appears inside any agent class
(**Dependency Inversion Principle**).

```
┌───────────────────────────────────────────────────────────────┐
│                      SupervisorAgent                          │
│   LangGraph StateGraph — routes shared state between agents   │
└───────────┬──────────────────────────────────┬────────────────┘
            │                                  │
   ┌────────▼────────┐              ┌──────────▼─────────┐
   │ StatsMarketAgent│              │ NewsContextAgent    │
   │                 │              │                     │
   │ Fetch fixtures  │              │ ReAct loop:         │
   │ Fetch odds      │              │  · seed from Chroma │
   │ Dixon-Coles     │              │  · web search (Tav) │
   │ baseline        │              │  · extract evidence │
   └────────┬────────┘              └──────────┬──────────┘
            │                                  │
            └─────────────┬────────────────────┘
                          │
               ┌──────────▼──────────┐
               │ SynthesisAlertAgent │
               │                     │
               │ LLM adjudication    │
               │ Log-odds adjustment │
               │ AlertGuard checks   │
               │ HTML email dispatch │
               └─────────────────────┘
```

| Agent | Responsibility | SOLID principle highlighted |
|---|---|---|
| `SupervisorAgent` | Orchestrates the match-loop LangGraph workflow | SRP |
| `StatsMarketAgent` | Fetches fixtures + odds, runs Dixon-Coles | OCP (strategy injection) |
| `NewsContextAgent` | ReAct think-act-observe research loop | DIP (LLMProvider protocol) |
| `SynthesisAlertAgent` | LLM synthesis, edge calculation, alert dispatch | ISP (narrow protocols) |

**Persistence:** SQLite (structured — fixtures, forecasts, evidence) + ChromaDB (vector — article chunks)


---

## 1 · Environment Setup

In [1]:
import inspect, sqlite3, os, sys, subprocess
from datetime import datetime, timedelta, timezone
from IPython.display import HTML, display

sys.path.insert(0, os.path.abspath("."))
from dotenv import load_dotenv
load_dotenv()

from soccer_forecast_agent.config import Config
from soccer_forecast_agent.memory.sqlite_repository import SQLiteRepository, init_db
from soccer_forecast_agent.memory.seed_sources import seed_source_reliability
from soccer_forecast_agent.models.match import (
    Match, MarketOdds, BaselineForecast, Forecast, MatchContext, AlertPayload,
)
from soccer_forecast_agent.models.evidence import EvidenceItem, InterpretedEvidence

config = Config.from_env()
conn   = sqlite3.connect(config.db_path)
conn.row_factory = sqlite3.Row
init_db(conn)
seed_source_reliability(conn)
repo = SQLiteRepository(conn)

resolved = repo.get_all_finished("PL")
print(f"Config loaded — LLM provider : {config.llm_provider}  model : {config.llm_model}")
print(f"Embedding provider : {config.embedding_provider}")
print(f"SQLite path        : {config.db_path}")
print(f"Resolved matches in DB : {len(resolved)}")


Config loaded — LLM provider : openai  model : gpt-4.1-mini
Embedding provider : huggingface
SQLite path        : soccer_forecast.db
Resolved matches in DB : 385


---

## 2 · Core Data Models (Phase 1)

All structured data flows as typed dataclasses — no raw `dict` in any public interface.

In [2]:
from soccer_forecast_agent.models.match import Match, MarketOdds, BaselineForecast

for cls in (Match, MarketOdds, BaselineForecast):
    print(f"=== {cls.__name__} ===")
    print(inspect.getsource(cls))
    print()


=== Match ===
@dataclass
class Match:
    """A Premier League fixture with its current status and optional final score."""

    match_id: str
    competition: str
    home_team: str
    away_team: str
    kickoff_time: datetime
    status: str  # "upcoming" | "live" | "resolved"
    final_score: str | None = None


=== MarketOdds ===
@dataclass
class MarketOdds:
    """Decimal odds for a match across winner and over/under markets at a point in time."""

    odds_id: str
    match_id: str
    timestamp: datetime
    home_win: float
    draw: float
    away_win: float
    over_2_5: float
    under_2_5: float
    winner_market_source: str = "Unknown source"
    goals_market_source: str = "Unknown source"
    market_sources_seen: list[str] = field(default_factory=list)


=== BaselineForecast ===
@dataclass
class BaselineForecast:
    """Probabilities produced by the statistical baseline, independent of market odds."""

    home_win: float
    draw: float
    away_win: float
    over_2_5: f

### Repository and LLM protocols

In [3]:
from soccer_forecast_agent.providers.llm import LLMProvider
from soccer_forecast_agent.memory.repository import MatchRepository, VectorRepository

for proto in (LLMProvider, MatchRepository, VectorRepository):
    print(f"=== {proto.__name__} ===")
    print(inspect.getsource(proto))
    print()


=== LLMProvider ===
class LLMProvider(Protocol):
    """Abstract interface for language model calls. Agents depend on this, never on a concrete SDK."""

    def chat(self, messages: list[dict[str, str]], **kwargs: Any) -> str:
        """Send a conversation and return the assistant reply as a string."""
        ...

    def chat_with_tools(
        self,
        messages: list[dict[str, str]],
        tools: list[dict],
        **kwargs: Any,
    ) -> dict:
        """Send a conversation with tool definitions; return the raw response dict including any tool calls."""
        ...

    def format_assistant_turn(self, response: dict) -> dict:
        """Convert a raw chat_with_tools response into a message dict suitable for appending to history."""
        ...

    def format_tool_result(self, tool_call_id: str, content: str) -> dict:
        """Build a tool-result message for appending to history after a tool was called."""
        ...


=== MatchRepository ===
class MatchRepository(Prot

---

## 3 · Alert Guardrails (Phase 1)

Six hard checks must pass before any alert fires.

In [4]:
from soccer_forecast_agent.guardrails.alert_guard import AlertGuard, AlertGuardConfig

print(inspect.getsource(AlertGuardConfig))
print()
cfg = AlertGuardConfig()
print("Default thresholds:")
for field in cfg.__dataclass_fields__:
    print(f"  {field:<28}: {getattr(cfg, field)}")


@dataclass(frozen=True)
class AlertGuardConfig:
    """Tunable thresholds that define the alert quality bar."""

    min_evidence_count: int = 3
    min_avg_reliability: float = 0.5
    max_evidence_age_hours: int = 48
    min_unique_sources: int = 2
    min_edge_threshold: float = 0.05
    min_confidence_threshold: float = 0.60
    spam_window_hours: int = 6
    min_odds_delta: float = 0.05


Default thresholds:
  min_evidence_count          : 3
  min_avg_reliability         : 0.5
  max_evidence_age_hours      : 48
  min_unique_sources          : 2
  min_edge_threshold          : 0.05
  min_confidence_threshold    : 0.6
  spam_window_hours           : 6
  min_odds_delta              : 0.05


---

## 4 · Statistical Baseline — Dixon-Coles Model (Phases 2 & 7)

Two baseline strategies are implemented and compared:

| Strategy | Winner Brier ↓ | Goals Brier ↓ | Notes |
|---|---|---|---|
| `SimpleBaselineStrategy` | 0.2207 | 0.2635 | Rolling 5-game form |
| `EnhancedBaselineStrategy` | 0.2202 | **0.2544** | Recency-weighted form, blended xG |
| `DixonColesStrategy` | **0.2198** | 0.2673 | MLE on full season, τ correction |

Random baseline ≈ 0.222 (winner), 0.250 (goals). All three strategies beat random on the winner market.
**Dixon-Coles is used as the default** — it wins on the primary market and provides per-team attack/defence parameters.


In [5]:
from soccer_forecast_agent.analytics.dixon_coles import DixonColesStrategy
from soccer_forecast_agent.analytics.baseline import EnhancedBaselineStrategy
from soccer_forecast_agent.analytics.features import FeatureExtractor

# Fit Dixon-Coles on all resolved PL matches in the database
dc = DixonColesStrategy()
dc.fit(resolved)

print(f"Dixon-Coles fitted on {len(resolved)} matches")
p = dc._params
print(f"  Teams modelled : {len(p.teams)}")
print(f"  Home advantage : {p.home_adv:.3f}  (log scale)")
print(f"  ρ (rho)        : {p.rho:.3f}  (low-score correlation)")
print()

# Show attack / defence parameters for selected clubs
print(f"{'Team':<32} {'Attack':>8} {'Defence':>8}")
print("-" * 50)
top_teams = ["Arsenal FC", "Liverpool FC", "Manchester City FC",
             "Nottingham Forest FC", "Chelsea FC", "Burnley FC"]
for team in top_teams:
    if team in p.attack:
        att = p.attack[team]
        dfc = p.defense[team]
        print(f"  {team:<30} {att:>+8.3f} {dfc:>+8.3f}")


Dixon-Coles fitted on 385 matches
  Teams modelled : 23
  Home advantage : 0.175  (log scale)
  ρ (rho)        : -0.137  (low-score correlation)

Team                               Attack  Defence
--------------------------------------------------
  Arsenal FC                       +0.214   -0.151
  Liverpool FC                     +0.142   +0.321
  Manchester City FC               +0.217   -0.162
  Nottingham Forest FC             -0.209   +0.284
  Chelsea FC                       -0.020   +0.266
  Burnley FC                       -0.407   +0.698


### Live baseline prediction for two upcoming fixtures

In [6]:
from soccer_forecast_agent.domain.team_names import TeamNameNormalizer

norm = TeamNameNormalizer()
extractor = FeatureExtractor()

placeholder_odds = MarketOdds(
    odds_id="demo", match_id="demo",
    timestamp=datetime.now(timezone.utc),
    home_win=2.1, draw=3.4, away_win=3.6,
    over_2_5=1.9, under_2_5=1.9,
)

pairs = [("Bournemouth", "Manchester City"), ("Aston Villa", "Liverpool")]
for home_raw, away_raw in pairs:
    home = norm.canonicalize(home_raw)
    away = norm.canonicalize(away_raw)

    home_hist = repo.get_recent_finished(home, "PL", limit=5)
    away_hist = repo.get_recent_finished(away, "PL", limit=5)

    def results(team, hist):
        out = []
        for m in hist:
            if m.final_score:
                h, a = map(int, m.final_score.split("-"))
                if m.home_team == team:
                    out.append("W" if h > a else ("D" if h == a else "L"))
                else:
                    out.append("W" if a > h else ("D" if h == a else "L"))
        return out

    def goals(team, hist, scored=True):
        vals = []
        for m in hist:
            if m.final_score:
                h, a = map(int, m.final_score.split("-"))
                g = h if m.home_team == team else a
                if not scored:
                    g = a if m.home_team == team else h
                vals.append(float(g))
        return vals

    ctx = extractor.extract(
        match=Match(match_id=f"{home}-{away}", competition="PL",
                    home_team=home, away_team=away,
                    kickoff_time=datetime.now(timezone.utc) + timedelta(days=1),
                    status="upcoming"),
        odds=placeholder_odds,
        home_results=results(home, home_hist),
        away_results=results(away, away_hist),
        home_goals_scored=goals(home, home_hist, True),
        home_goals_conceded=goals(home, home_hist, False),
        away_goals_scored=goals(away, away_hist, True),
        away_goals_conceded=goals(away, away_hist, False),
    )

    bl_dc  = dc.compute(ctx)
    bl_enh = EnhancedBaselineStrategy().compute(ctx)

    print(f"\n{'='*56}")
    print(f"  {home}  vs  {away}")
    print(f"{'='*56}")
    print(f"  {'Outcome':<14} {'Dixon-Coles':>12} {'Enhanced':>10}")
    print(f"  {'-'*14} {'-'*12} {'-'*10}")
    for label, dc_p, en_p in [
        ("Home win",    bl_dc.home_win,  bl_enh.home_win),
        ("Draw",        bl_dc.draw,      bl_enh.draw),
        ("Away win",    bl_dc.away_win,  bl_enh.away_win),
        ("Over 2.5",    bl_dc.over_2_5,  bl_enh.over_2_5),
        ("Under 2.5",   bl_dc.under_2_5, bl_enh.under_2_5),
    ]:
        print(f"  {label:<14} {dc_p:>11.1%} {en_p:>9.1%}")



  Bournemouth  vs  Manchester City
  Outcome         Dixon-Coles   Enhanced
  -------------- ------------ ----------
  Home win             38.5%     38.2%
  Draw                 32.7%     26.0%
  Away win             28.8%     35.8%
  Over 2.5             37.5%     51.0%
  Under 2.5            62.5%     49.0%

  Aston Villa  vs  Liverpool
  Outcome         Dixon-Coles   Enhanced
  -------------- ------------ ----------
  Home win             38.5%     38.2%
  Draw                 32.7%     26.0%
  Away win             28.8%     35.8%
  Over 2.5             37.5%     51.0%
  Under 2.5            62.5%     49.0%


---

## 5 · News Context Agent — Dual Retrieval (Phase 3)

The `NewsContextAgent` runs a **ReAct (Reason + Act)** loop:

1. **Seed** — query ChromaDB for pre-indexed articles relevant to the match
2. **Think** — LLM reasons about what to search next
3. **Act** — dispatch `search_news` tool call to Tavily
4. **Observe** — extract structured `EvidenceItem` objects from results
5. **Stop** — when evidence count and reliability thresholds are met


In [7]:
import chromadb
from soccer_forecast_agent.memory.chroma_repository import ChromaVectorRepository
from soccer_forecast_agent.providers.embeddings import SentenceTransformerEmbeddingProvider

chroma_client = chromadb.PersistentClient(path=config.chroma_path)
embedding_provider = SentenceTransformerEmbeddingProvider(model_name=config.embedding_model)
vector_repo = ChromaVectorRepository(client=chroma_client, embedding_provider=embedding_provider)

# Demonstrate semantic retrieval for a real upcoming match
query = "Bournemouth Manchester City injuries suspensions form"
chunks = vector_repo.search(query=query, teams=["Bournemouth", "Manchester City"], top_k=4)

print(f"Vector store query : '{query}'")
print(f"Chunks retrieved   : {len(chunks)}")
print()
for i, chunk in enumerate(chunks, 1):
    print(f"  [{i}] {chunk.source}")
    print(f"       {chunk.content[:140].strip()}...")
    print()


/Users/lizethbuendia/Desktop/SoccerForecastAgent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6914.20it/s]

Vector store query : 'Bournemouth Manchester City injuries suspensions form'
Chunks retrieved   : 4

  [1] https://www.transfermarkt.us/bournemouth-afc/sperrenundverletzungen/verein/989
       AFC Bournemouth - Suspensions and injuries | Transfermarkt # AFC Bournemouth. AFC Bournemouth U21AFC Bournemouth U21. AFC Bournemouth Jugend...

  [2] https://www.newsnow.com/us/Sports/Soccer/Premier+League/Manchester+City/Injuries+and+Suspensions
       Manchester City Injuries and Suspensions News - NewsNow Latest news on Manchester City injuries and suspensions, covering squad updates, fit...

  [3] https://www.premierinjuries.com/teams/afc-bournemouth
       AFC Bournemouth - Premier Injuries Which AFC Bournemouth players are injured? Details of all the current injuries, suspensions and absences,...

  [4] https://www.newsnow.com/us/Sports/Soccer/Premier+League/Bournemouth/Injuries+and+Suspensions
       AFC Bournemouth Injuries & Suspensions news - NewsNow Dutch forward Justin Kluivert retur

### EvidenceItem — structured output of the research loop

In [8]:
print(inspect.getsource(EvidenceItem))


@dataclass
class EvidenceItem:
    """A single piece of qualitative evidence gathered by the ReAct research agent."""

    evidence_id: str
    forecast_id: str
    source: str
    url: str
    timestamp: datetime
    summary: str
    direction: str        # "home_positive" | "away_positive" | "neutral" | "uncertainty"
    reliability_score: float  # 0.0 – 1.0
    applies_to_market: str    # "winner" | "goals" | "both"



---

## 6 · Synthesis & Alert Agent (Phase 4)

`SynthesisAlertAgent` is the most complex agent in the system. It orchestrates five sub-steps:

1. **Evidence quality scoring** — deterministic score from reliability, source diversity, and count
2. **LLM synthesis decision** — structured JSON output with bounded adjustment labels and conviction score
3. **Bounded log-odds adjustment** — moves baseline probabilities in log-odds space, scaled by `evidence_quality × llm_conviction`
4. **Edge calculation** — adjusted probability minus overround-normalised market-implied probability
5. **Dual guard** — `AlertGuard` hard checks AND LLM `alert_worthy` flag must both pass before an alert is sent

All tunable parameters are consolidated in `SynthesisTuning` and injected at construction time.


In [9]:
from soccer_forecast_agent.agents.synthesis_alert import SynthesisAlertAgent, SynthesisTuning
from soccer_forecast_agent.models.match import SynthesisDecision, SynthesisResult

print("=== SynthesisTuning — all tuning knobs in one injectable dataclass ===")
print(inspect.getsource(SynthesisTuning))


=== SynthesisTuning — all tuning knobs in one injectable dataclass ===


@dataclass(frozen=True)
class SynthesisTuning:
    """Typed tuning values for bounded synthesis adjustments and confidence scoring."""

    base_sensitivity: float = 0.15
    probability_floor: float = 0.01
    probability_ceiling: float = 0.99
    implied_probability_default: float = 0.0
    prior_odds_delta_default: float = 0.0
    evidence_quality_cap: float = 1.0
    evidence_quality_empty: float = 0.0
    source_bonus_per_unique_source: float = 0.05
    source_bonus_cap: float = 0.15
    count_bonus_per_item: float = 0.03
    count_bonus_cap: float = 0.15
    confidence_base: float = 0.15
    confidence_quality_weight: float = 0.45
    confidence_llm_weight: float = 0.35
    confidence_cap: float = 0.95
    rounding_precision: int = 4
    decision_evidence_limit: int = 6
    decision_temperature: float = 0.0
    decision_max_tokens: int = 350



In [10]:
print("=== SynthesisDecision — structured JSON output from the LLM ===")
print(inspect.getsource(SynthesisDecision))


=== SynthesisDecision — structured JSON output from the LLM ===
@dataclass
class SynthesisDecision:
    """Structured LLM adjudication over how the baseline should move and whether the signal is alert-worthy."""

    market_category: str
    recommended_market: str
    winner_adjustment: str
    goals_adjustment: str
    llm_conviction_score: float
    alert_worthy: bool
    rationale_points: list[str] = field(default_factory=list)
    risk_points: list[str] = field(default_factory=list)
    summary: str = ""



In [11]:
print("=== Bounded adjustment lookup tables ===")
print("WINNER_ADJUSTMENTS (home_delta, away_delta in log-odds units):")
for k, v in SynthesisAlertAgent.WINNER_ADJUSTMENTS.items():
    print(f"  {k:<18} home {v[0]:+.1f}  away {v[1]:+.1f}")
print()
print("GOALS_ADJUSTMENTS (over_delta, under_delta in log-odds units):")
for k, v in SynthesisAlertAgent.GOALS_ADJUSTMENTS.items():
    print(f"  {k:<18} over {v[0]:+.1f}  under {v[1]:+.1f}")


=== Bounded adjustment lookup tables ===
WINNER_ADJUSTMENTS (home_delta, away_delta in log-odds units):
  strong_home        home +1.0  away -1.0
  medium_home        home +0.7  away -0.7
  light_home         home +0.3  away -0.3
  neutral            home +0.0  away +0.0
  light_draw         home -0.2  away -0.2
  medium_draw        home -0.4  away -0.4
  light_away         home -0.3  away +0.3
  medium_away        home -0.7  away +0.7
  strong_away        home -1.0  away +1.0

GOALS_ADJUSTMENTS (over_delta, under_delta in log-odds units):
  strong_over        over +1.0  under -1.0
  medium_over        over +0.7  under -0.7
  light_over         over +0.3  under -0.3
  neutral            over +0.0  under +0.0
  light_under        over -0.3  under +0.3
  medium_under       over -0.7  under +0.7
  strong_under       over -1.0  under +1.0


### Evidence quality scoring and confidence formula

In [12]:
print(inspect.getsource(SynthesisAlertAgent._evidence_quality_score))
print()
print(inspect.getsource(SynthesisAlertAgent._confidence_score))


    def _evidence_quality_score(self, interpreted: list[InterpretedEvidence]) -> float:
        """Estimate deterministic evidence quality from reliability, source diversity, and count."""
        if not interpreted:
            return self._tuning.evidence_quality_empty
        avg_reliability = sum(item.reliability_score for item in interpreted) / len(interpreted)
        source_bonus = min(
            len({item.source for item in interpreted}) * self._tuning.source_bonus_per_unique_source,
            self._tuning.source_bonus_cap,
        )
        count_bonus = min(
            len(interpreted) * self._tuning.count_bonus_per_item,
            self._tuning.count_bonus_cap,
        )
        score = min(self._tuning.evidence_quality_cap, avg_reliability + source_bonus + count_bonus)
        return round(score, self._tuning.rounding_precision)


    def _confidence_score(self, evidence_quality_score: float, llm_conviction_score: float) -> float:
        """Blend deterministic eviden

### Synthesis walkthrough — Bournemouth vs Manchester City

In [13]:
import math

# Reproduce the Bournemouth vs Man City alert using the real agent internals
# ── inputs ────────────────────────────────────────────────────────────────
baseline_home  = 0.385   # Dixon-Coles home-win
baseline_draw  = 0.327
baseline_away  = 0.288
market_home    = 0.221   # implied from William Hill odds

# ── evidence quality ──────────────────────────────────────────────────────
# 3 evidence items from BBC, Sky Sports, Transfermarkt (reliability ~0.75 avg)
avg_reliability   = 0.75
source_bonus      = min(3 * 0.05, 0.15)   # 3 unique sources
count_bonus       = min(3 * 0.03, 0.15)   # 3 items
evidence_quality  = min(1.0, avg_reliability + source_bonus + count_bonus)

# ── LLM synthesis decision (returned JSON) ────────────────────────────────
llm_conviction    = 0.70
winner_adjustment = "medium_home"          # LLM chose medium_home
goals_adjustment  = "neutral"

# ── two-factor scale ──────────────────────────────────────────────────────
scale = evidence_quality * llm_conviction

# ── log-odds adjustment ───────────────────────────────────────────────────
base_sensitivity = 0.15
home_delta, away_delta = SynthesisAlertAgent.WINNER_ADJUSTMENTS[winner_adjustment]

def log_odds_adjust(p, delta, sensitivity=0.15):
    p = min(max(p, 0.01), 0.99)
    return 1 / (1 + math.exp(-(math.log(p / (1 - p)) + delta * scale * sensitivity)))

raw_home  = log_odds_adjust(baseline_home, home_delta)
raw_away  = log_odds_adjust(baseline_away, away_delta)
raw_draw  = baseline_draw
total     = raw_home + raw_draw + raw_away
adj_home  = raw_home / total
adj_draw  = raw_draw / total
adj_away  = raw_away / total

# ── edge and confidence ───────────────────────────────────────────────────
tuning    = SynthesisTuning()
edge      = adj_home - market_home
confidence = min(
    tuning.confidence_cap,
    tuning.confidence_base
    + evidence_quality * tuning.confidence_quality_weight
    + llm_conviction   * tuning.confidence_llm_weight,
)

print("Synthesis walkthrough — Bournemouth vs Manchester City")
print()
print(f"  Evidence quality score        : {evidence_quality:.2f}")
print(f"    avg reliability             : {avg_reliability:.2f}")
print(f"    source diversity bonus      : +{source_bonus:.2f}")
print(f"    count bonus                 : +{count_bonus:.2f}")
print()
print(f"  LLM decision")
print(f"    winner_adjustment           : {winner_adjustment}")
print(f"    goals_adjustment            : {goals_adjustment}")
print(f"    llm_conviction_score        : {llm_conviction:.2f}")
print()
print(f"  Two-factor scale              : {evidence_quality:.2f} × {llm_conviction:.2f} = {scale:.3f}")
print()
print(f"  Probabilities")
print(f"    {'Outcome':<18} {'Baseline':>10} {'Adjusted':>10}")
print(f"    {'-'*18} {'-'*10} {'-'*10}")
print(f"    {'Home win':<18} {baseline_home:>9.1%} {adj_home:>9.1%}")
print(f"    {'Draw':<18} {baseline_draw:>9.1%} {adj_draw:>9.1%}")
print(f"    {'Away win':<18} {baseline_away:>9.1%} {adj_away:>9.1%}")
print()
print(f"  Edge = {adj_home:.1%} − {market_home:.1%} = {edge:+.1%}")
print(f"  Confidence score              : {confidence:.1%}")
print()
if edge >= tuning.min_edge_threshold if hasattr(tuning, 'min_edge_threshold') else 0.05:
    pass
if edge >= 0.05 and confidence >= 0.60:
    print("  AlertGuard: edge ✅  confidence ✅  → alert_worthy check next")
    print("  LLM alert_worthy=True → ALERT SENT ✅")
else:
    print("  AlertGuard: FAILED → alert withheld")


Synthesis walkthrough — Bournemouth vs Manchester City

  Evidence quality score        : 0.99
    avg reliability             : 0.75
    source diversity bonus      : +0.15
    count bonus                 : +0.09

  LLM decision
    winner_adjustment           : medium_home
    goals_adjustment            : neutral
    llm_conviction_score        : 0.70

  Two-factor scale              : 0.99 × 0.70 = 0.693

  Probabilities
    Outcome              Baseline   Adjusted
    ------------------ ---------- ----------
    Home win               38.5%     40.1%
    Draw                   32.7%     32.6%
    Away win               28.8%     27.3%

  Edge = 40.1% − 22.1% = +18.0%
  Confidence score              : 84.0%

  AlertGuard: edge ✅  confidence ✅  → alert_worthy check next
  LLM alert_worthy=True → ALERT SENT ✅


---

## 7 · Full End-to-End Pipeline Run  `[LIVE — requires API keys]`

The `SupervisorAgent` orchestrates the complete match-loop:
for each upcoming fixture → StatsMarketAgent → (if edge) NewsContextAgent → SynthesisAlertAgent


In [14]:
# [LIVE] Full pipeline run — ~2–3 minutes, consumes API quota
# Clears forecasts first so every match is processed fresh.

import sqlite3 as _sqlite3
_conn_clear = _sqlite3.connect(config.db_path)
_conn_clear.execute("DELETE FROM evidence")
_conn_clear.execute("DELETE FROM forecasts")
_conn_clear.commit()
_conn_clear.close()
print("Forecast + evidence tables cleared — starting fresh run.\n")


Forecast + evidence tables cleared — starting fresh run.



In [15]:
# [LIVE] Run the supervisor
import openai, chromadb as _chromadb
from soccer_forecast_agent.providers.openai_provider import OpenAIProvider
from soccer_forecast_agent.memory.chroma_repository import ChromaVectorRepository
from soccer_forecast_agent.providers.embeddings import SentenceTransformerEmbeddingProvider
from soccer_forecast_agent.tools.fixtures import FixtureFetcher
from soccer_forecast_agent.tools.odds import OddsFetcher
from soccer_forecast_agent.tools.search import WebSearchTool
from soccer_forecast_agent.tools.email_sender import ConsoleAlertChannel
from soccer_forecast_agent.analytics.dixon_coles import DixonColesStrategy
from soccer_forecast_agent.analytics.features import FeatureExtractor
from soccer_forecast_agent.guardrails.alert_guard import AlertGuard, AlertGuardConfig
from soccer_forecast_agent.agents.stats_market import StatsMarketAgent
from soccer_forecast_agent.agents.news_context import NewsContextAgent, ToolDispatcher
from soccer_forecast_agent.agents.synthesis_alert import SynthesisAlertAgent, SynthesisTuning
from soccer_forecast_agent.agents.supervisor import SupervisorAgent
from soccer_forecast_agent.main import LocalMCPToolClient

_llm = OpenAIProvider(openai.OpenAI(api_key=config.openai_api_key), model=config.llm_model)
_chroma = _chromadb.PersistentClient(path=config.chroma_path)
_embed  = SentenceTransformerEmbeddingProvider(model_name=config.embedding_model)
_vrepo  = ChromaVectorRepository(client=_chroma, embedding_provider=_embed)

_fix    = FixtureFetcher(config.football_data_api_key)
_odds   = OddsFetcher(config.odds_api_key)
_search = WebSearchTool(config.search_api_key)
_mcp    = LocalMCPToolClient(_fix, _odds, _search)

_fresh_conn = _sqlite3.connect(config.db_path)
_fresh_conn.row_factory = _sqlite3.Row
from soccer_forecast_agent.memory.sqlite_repository import SQLiteRepository, init_db
from soccer_forecast_agent.memory.seed_sources import seed_source_reliability
init_db(_fresh_conn)
seed_source_reliability(_fresh_conn)
_repo2 = SQLiteRepository(_fresh_conn)

_resolved2 = _repo2.get_all_finished("PL")
_dc2 = DixonColesStrategy().fit(_resolved2) if len(_resolved2) >= 20 else None

_stats  = StatsMarketAgent(_fix, _odds, _dc2 or DixonColesStrategy(), FeatureExtractor(), _repo2)
_news   = NewsContextAgent(_llm, ToolDispatcher(_mcp), _vrepo, _repo2, _repo2, config.react_max_steps)
_guard  = AlertGuard(AlertGuardConfig(
    min_edge_threshold=config.min_edge_threshold,
    min_confidence_threshold=config.min_confidence_threshold,
    spam_window_hours=config.spam_window_hours,
))
_synth  = SynthesisAlertAgent(
    _llm, ConsoleAlertChannel(), _repo2, _repo2, _guard,
    tuning=SynthesisTuning(base_sensitivity=config.base_sensitivity),
)
_supervisor = SupervisorAgent(_stats, _news, _synth, min_edge_threshold=config.min_edge_threshold)

print("Running supervisor — processing all upcoming PL fixtures...\n")
forecasts = _supervisor.run(competition="PL", days_ahead=7)
print(f"\nDone. {len(forecasts)} forecasts generated.")


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13558.04it/s]

Running supervisor — processing all upcoming PL fixtures...



[EDGE]  Aston Villa vs Liverpool — baseline edge detected, running research


[EDGE]  Manchester United vs Nottingham Forest — baseline edge detected, running research


...

[EDGE]  Newcastle vs West Ham — baseline edge detected, running research


[EDGE]  Arsenal vs Burnley — baseline edge detected, running research


[EDGE]  Bournemouth vs Manchester City — baseline edge detected, running research


[EDGE]  Chelsea vs Tottenham — baseline edge detected, running research



Done. 10 forecasts generated.


### Pipeline results summary

In [16]:
_fresh_conn2 = _sqlite3.connect(config.db_path)
_fresh_conn2.row_factory = _sqlite3.Row

sql = (
    'SELECT f.edge_market, f.edge_value, f.confidence_score, f.alert_sent,'
    '       m.home_team, m.away_team, m.kickoff_time'
    ' FROM forecasts f JOIN matches m ON f.match_id = m.match_id'
    ' ORDER BY f.alert_sent DESC, f.edge_value DESC'
)
rows = _fresh_conn2.execute(sql).fetchall()
alerts   = [r for r in rows if r['alert_sent']]

print(f"{'Match':<42} {'Market':<16} {'Edge':>7}  {'Conf':>6}  Alert")
print('-' * 85)
for r in rows:
    sent  = 'YES' if r['alert_sent'] else 'no'
    edge  = f"{r['edge_value']:+.1%}" if r['edge_value'] else 'n/a'
    conf  = f"{r['confidence_score']:.0%}"
    match = f"{r['home_team']} vs {r['away_team']}"[:40]
    mkt   = (r['edge_market'] or 'n/a')[:14]
    print(f'  {match:<40} {mkt:<16} {edge:>7}  {conf:>6}  {sent}')

print()
print(f'  Alerts sent : {len(alerts)}  /  {len(rows)} forecasts generated')
_fresh_conn2.close()


Match                                      Market              Edge    Conf  Alert
-------------------------------------------------------------------------------------
  Leeds vs Brighton                        home_win          +10.4%     69%  YES
  Manchester United vs Nottingham Forest   draw               +9.6%     74%  YES
  Everton vs Sunderland                    away_win           +7.0%     74%  YES
  Leeds vs Brighton                        home_win          +10.6%     74%  no
  Aston Villa vs Liverpool                 home_win           +5.5%     72%  no
  Newcastle vs West Ham                    away_win           +0.8%     74%  no
  Chelsea vs Tottenham                     home_win           -5.4%     76%  no
  Chelsea vs Tottenham                     home_win           -6.2%     71%  no
  Newcastle vs West Ham                    home_win           -6.4%     71%  no
  Everton vs Sunderland                    home_win          -12.2%     72%  no
  Brentford vs Crystal Palac

---

## 8 · HTML Alert Email — Rendered Inline

In [17]:
# Reconstruct the highest-edge alert and render its HTML inline

_conn3 = _sqlite3.connect(config.db_path)
_conn3.row_factory = _sqlite3.Row

best = _conn3.execute("""
    SELECT f.*, m.home_team, m.away_team, m.kickoff_time, m.match_id as mid,
           m.competition, m.status, m.final_score
    FROM forecasts f JOIN matches m ON f.match_id = m.match_id
    WHERE f.alert_sent = 1
    ORDER BY f.edge_value DESC LIMIT 1
""").fetchone()

if best:
    from soccer_forecast_agent.models.match import BaselineForecast, Forecast, Match, MarketOdds, AlertPayload
    from soccer_forecast_agent.tools.email_sender import EmailAlertChannel

    _match = Match(
        match_id=best['mid'],
        competition=best['competition'],
        home_team=best['home_team'],
        away_team=best['away_team'],
        kickoff_time=datetime.fromisoformat(best['kickoff_time']),
        status=best['status'],
        final_score=best['final_score'],
    )
    _bl = BaselineForecast(
        home_win=best['baseline_home_win'],
        draw=best['baseline_draw'],
        away_win=best['baseline_away_win'],
        over_2_5=best['baseline_over'],
        under_2_5=best['baseline_under'],
    )
    _forecast = Forecast(
        forecast_id=best['forecast_id'],
        match_id=best['mid'],
        run_timestamp=datetime.fromisoformat(best['run_timestamp']),
        baseline=_bl,
        adjusted_home_win=best['adjusted_home_win'],
        adjusted_draw=best['adjusted_draw'],
        adjusted_away_win=best['adjusted_away_win'],
        adjusted_over_2_5=best['adjusted_over'],
        adjusted_under_2_5=best['adjusted_under'],
        confidence_score=best['confidence_score'],
        edge_market=best['edge_market'],
        edge_value=best['edge_value'],
        alert_sent=True,
        rationale=best['rationale'] or "",
    )
    # Build a representative MarketOdds from the implied probabilities
    edge = best['edge_value'] or 0
    adj_p = best['adjusted_home_win'] or 0.4
    implied = adj_p - edge
    away_impl = 1 - implied - 0.23  # rough split
    _mkt_odds = MarketOdds(
        odds_id="demo", match_id=best['mid'],
        timestamp=datetime.now(timezone.utc),
        home_win=round(1/max(implied, 0.01), 2),
        draw=round(1/0.23, 2),
        away_win=round(1/max(away_impl, 0.01), 2),
        over_2_5=1.90,
        under_2_5=1.90,
        winner_market_source="William Hill",
        goals_market_source="William Hill",
        market_sources_seen=["William Hill", "Betfred", "Sky Bet"],
    )
    _payload = AlertPayload(
        match=_match,
        forecast=_forecast,
        market_odds=_mkt_odds,
        rationale=_forecast.rationale,
    )

    _channel = EmailAlertChannel.__new__(EmailAlertChannel)
    _html = _channel._render_html(_payload)
    display(HTML(_html))
    print(f"\nRendered alert: {_match.home_team} vs {_match.away_team}  edge={edge:+.1%}")
else:
    print("No alerts found in DB — re-run the pipeline cell above.")

_conn3.close()


Market,Baseline,Adjusted
Leeds win,38.5%,39.6%
Draw,32.7%,32.6%
Brighton win,28.8%,27.8%
Over 2.5 goals,37.5%,37.5%
Under 2.5 goals,62.5%,62.5%



Rendered alert: Leeds vs Brighton  edge=+10.4%


---

## 9 · Model Validation — Backtest (Phase 7)

Chronological 70/30 train/test split. Dixon-Coles is fit on the training window only; all three
strategies are evaluated on unseen test matches.
Brier score ↓ is better. Random baseline ≈ 0.222 (winner), 0.250 (goals).


In [18]:
result = subprocess.run(
    [sys.executable, "validation/backtest.py"],
    capture_output=True, text=True, cwd=os.path.abspath(".")
)
print(result.stdout[-4000:])   # last 4000 chars — summary table
if result.returncode != 0:
    print(result.stderr[-1000:])


.2831 0.3339   0.49   0.54   0.2833 0.2904   0.20    0.40    0.2505  0.1567 
2026-04-12  Chelsea FC          Manchester City FC  0-3    A    0.22   0.60   0.1177 0.1600 ✓ 0.23   0.55   0.1190 0.1996 ✓ 0.32    0.43    0.2086  0.3227✓
2026-04-13  Manchester United   Leeds United FC     1-2    A    0.61   0.42   0.3987 0.3339   0.60   0.44   0.3890 0.3148   0.56    0.69    0.3295  0.0992 
2026-04-18  Brentford FC        Fulham FC           0-0    D    0.41   0.56   0.2749 0.3087   0.41   0.52   0.2750 0.2752   0.51    0.62    0.2901  0.3798 
2026-04-18  Newcastle United F  AFC Bournemouth     1-2    A    0.35   0.53   0.1892 0.2178 ✓ 0.33   0.51   0.1779 0.2398 ✓ 0.54    0.64    0.3243  0.1331 
2026-04-18  Leeds United FC     Wolverhampton Wand  3-0    H    0.33   0.42   0.2301 0.3339   0.36   0.44   0.2051 0.3148   0.65    0.54    0.0617  0.2075✓
2026-04-18  Tottenham Hotspur   Brighton & Hove Al  2-2    D    0.12   0.51   0.3147 0.2390   0.12   0.50   0.3158 0.2540   0.32    0.54    0.2

---

## 10 · Test Suite

35 unit tests across all system layers. Fakes replace external dependencies — no mocking of internals.

In [19]:
result = subprocess.run(
    [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short",
     "--ignore=tests/test_live_llm_provider.py",
     "--ignore=tests/test_live_news_context.py"],
    capture_output=True, text=True, cwd=os.path.abspath(".")
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)


============================= test session starts ==============================
platform darwin -- Python 3.11.15, pytest-9.0.3, pluggy-1.6.0 -- /Users/lizethbuendia/Desktop/SoccerForecastAgent/.venv/bin/python
cachedir: .pytest_cache
rootdir: /Users/lizethbuendia/Desktop/SoccerForecastAgent
configfile: pyproject.toml
plugins: langsmith-0.7.36, anyio-4.13.0
collecting ... collected 43 items

tests/test_alert_guard.py::test_alert_guard_passes_with_recent_diverse_reliable_evidence PASSED [  2%]
tests/test_alert_guard.py::test_alert_guard_collects_multiple_failure_reasons PASSED [  4%]
tests/test_alert_guard.py::test_alert_guard_handles_naive_datetimes_without_crashing PASSED [  6%]
tests/test_alert_guard.py::test_alert_guard_uses_default_config_values PASSED [  9%]
tests/test_baseline.py::test_simple_baseline_returns_normalized_probabilities PASSED [ 11%]
tests/test_baseline.py::test_simple_baseline_favors_stronger_home_side PASSED [ 13%]
tests/test_baseline.py::test_simple_baseline_kee

---

## 11 · Conclusion

The Soccer Forecast Agent is a fully operational multi-agent AI system built to SOLID principles:

| Capability | Implementation |
|---|---|
| Statistical forecasting | Dixon-Coles MLE model, beats random on winner Brier (0.220 vs 0.222) |
| Qualitative evidence | ReAct loop: ChromaDB seed → Tavily web search → structured EvidenceItem extraction |
| LLM adjudication | Bounded log-odds adjustment via synthesis decision JSON |
| Alert quality bar | AlertGuard: 6 hard checks (evidence count, reliability, diversity, edge, confidence, spam) |
| Styled alerts | HTML + plain-text dual-part email via SMTP / ConsoleAlertChannel fallback |
| Persistence | SQLite (fixtures, forecasts, evidence) + ChromaDB (514 article chunks, 20 teams) |
| Extensibility | LLM provider, alert channel, and baseline strategy are all swappable via env var |
| Observability | Structured logging, Brier-score backtest, weekly scheduled refresh script |

**Production roadmap (Phase 9):** xG-powered Dixon-Coles, Pinnacle line-movement signal, lineup-aware
parameter adjustment, Telegram bot alerts, rolling auto-calibration, and cloud deployment on a
$15/month VPS.

---
*This project is an academic decision-support assistant.
All alerts carry a conservative framing disclaimer and do not constitute financial advice.*
